# Task 5: Custom Batch Normalization and Layer Normalization Forward/Backward Propagation

**Objective:** Stablize internal covariate shifts by building structural normalization layers with manual gradient derivations, bypassing PyTorch Autograd.

In [ ]:
import torch
import numpy as np

class CustomBatchNormFunction:
    @staticmethod
    def forward(x, gamma, beta, running_mean, running_var, eps=1e-5, training=True, momentum=0.1):
        # x shape: (N, D)
        if training:
            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)
            
            # Normalize
            x_hat = (x - mean) / torch.sqrt(var + eps)
            
            # Update running stats
            running_mean.copy_((1 - momentum) * running_mean + momentum * mean)
            running_var.copy_((1 - momentum) * running_var + momentum * var * x.shape[0] / (x.shape[0] - 1))
            
            # Cache variables for backprop
            cache = (x, x_hat, mean, var, gamma, eps)
        else:
            x_hat = (x - running_mean) / torch.sqrt(running_var + eps)
            cache = None
            
        y = gamma * x_hat + beta
        return y, cache
        
    @staticmethod
    def backward(dy, cache):
        # dy shape: (N, D)
        x, x_hat, mean, var, gamma, eps = cache
        N, D = dy.shape
        
        # Gradients w.r.t parameters
        dgamma = torch.sum(dy * x_hat, dim=0)
        dbeta = torch.sum(dy, dim=0)
        
        # Intermediate gradients w.r.t normalization formula
        dx_hat = dy * gamma
        dvar = torch.sum(dx_hat * (x - mean) * -0.5 * (var + eps)**(-1.5), dim=0)
        dmean = torch.sum(dx_hat * -1.0 / torch.sqrt(var + eps), dim=0) + dvar * torch.mean(-2.0 * (x - mean), dim=0)
        
        # Final inputs gradient
        dx = dx_hat / torch.sqrt(var + eps) + dvar * 2.0 * (x - mean) / N + dmean / N
        return dx, dgamma, dbeta

class CustomLayerNormFunction:
    @staticmethod
    def forward(x, gamma, beta, eps=1e-5):
        # x shape: (N, D) - normalization is done per sample across features D
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        x_hat = (x - mean) / torch.sqrt(var + eps)
        y = gamma * x_hat + beta
        
        cache = (x, x_hat, mean, var, gamma, eps)
        return y, cache
        
    @staticmethod
    def backward(dy, cache):
        x, x_hat, mean, var, gamma, eps = cache
        N, D = dy.shape
        
        # Gradients w.r.t parameters
        dgamma = torch.sum(dy * x_hat, dim=0)
        dbeta = torch.sum(dy, dim=0)
        
        # Inputs gradients via Layer Normalization derivatives
        dx_hat = dy * gamma
        dvar = torch.sum(dx_hat * (x - mean) * -0.5 * (var + eps)**(-1.5), dim=-1, keepdim=True)
        dmean = torch.sum(dx_hat * -1.0 / torch.sqrt(var + eps), dim=-1, keepdim=True) + dvar * torch.mean(-2.0 * (x - mean), dim=-1, keepdim=True)
        
        dx = dx_hat / torch.sqrt(var + eps) + dvar * 2.0 * (x - mean) / D + dmean / D
        return dx, dgamma, dbeta

In [ ]:
# Verify custom implementation using PyTorch Autograd
N, D = 16, 8
x = torch.randn(N, D, requires_grad=True)
gamma = torch.ones(D, requires_grad=True)
beta = torch.zeros(D, requires_grad=True)

# Test Batch Norm
running_mean = torch.zeros(D)
running_var = torch.ones(D)

# PyTorch output
pt_bn = torch.nn.BatchNorm1d(D, eps=1e-5, momentum=0.1)
pt_bn.weight.data = gamma.data.clone()
pt_bn.bias.data = beta.data.clone()
pt_y = pt_bn(x)

# Custom output
custom_y, cache = CustomBatchNormFunction.forward(x, gamma, beta, running_mean, running_var, eps=1e-5, training=True)

print("BatchNorm Forward Difference:", torch.abs(pt_y - custom_y).max().item())

# Backward Validation
dy = torch.randn(N, D)
pt_y.backward(dy)

dx_custom, dgamma_custom, dbeta_custom = CustomBatchNormFunction.backward(dy, cache)

print("BatchNorm Input Gradient Difference:", torch.abs(x.grad - dx_custom).max().item())
print("BatchNorm Gamma Gradient Difference:", torch.abs(gamma.grad - dgamma_custom).max().item())
print("BatchNorm Beta Gradient Difference:", torch.abs(beta.grad - dbeta_custom).max().item())

assert torch.abs(x.grad - dx_custom).max().item() < 1e-5, "Failed Batch Norm Gradient Match!"
print("Success: Batch Normalization custom gradients match PyTorch Autograd!")